# P3 variant — huan luyen + eval da nhanh trong MOT lan chay

Doi `VARIANT` o Cell 2 roi **Run All** (~3h). Khong phai upload checkpoint, khong phai mo
notebook thu hai. Can: GPU + Add Input `forget-mi-data` va `forget-mi-models-full`.

| VARIANT | Khac biet | Trang thai |
|---|---|---|
| `f5` | `loku_subtract_scale=1.5` | da chay xong 13/8 |
| `f6` | **`loku_subtract_scale=1.0`** (gia tri chuan LoKU) | can chay |

## F6 la gi

`f6 = f5` doi **dung mot khoa**: `loku_subtract_scale` **1.5 -> 1.0**. Day la he so gamma
trong `W* = W - gamma*B*A*`, tuc muc tru FILA o nhanh VAN BAN. 1.0 = tru dung phan Fisher
uoc luong (gia tri chuan cua LoKU); 1.5 = tru qua da 50%, mot con so dat ra khi dung luoi
F1-F4 chu khong co co so ly thuyet.

Ba ly do bo 1.5:

1. **Vo tac dung** — F2 vs F3 khac nhau dung khoa nay: Df-AUC 0.603 vs 0.590, Dt-AUC 0.694
   vs 0.692, Forget-CE 3.317 vs 3.477. Chenh trong nhieu.
2. **Gay tran FP16 o nhanh van ban** — f3 va f5 deu cho logits_txt NaN toan bo khi eval
   FP16, phai chay FP32 moi do duoc. f4 cung gamma=1.5 ma khong tran -> qua bom hen gio.
3. **Kho bao ve** — khong tra loi duoc cau "vi sao 1.5", trong khi 1.0 la gia tri bai bao
   quy dinh.

Ky vong: **F6 ~ F5** (Df-AUC quanh 0.52, Dt-AUC quanh 0.67). Neu dung vay thi chot F6 —
cau hinh sach, do duoc o FP16 nhu moi phuong phap khac, khong can chu thich rieng.
Neu F6 kem han thi giu F5, va gamma=1.5 tro thanh mot phat hien that.

Vi so tham so KHONG phu thuoc gamma, F6 phai ra **dung 1 488 896 tham so** nhu F5. Lech la
cau hinh chua vao dung -> tat ngay.

## Cau hinh day du (moi lambda)

| | F3 | F4 | F5 | **F6** |
|---|---:|---:|---:|---:|
| w_UR / w_UU / w_MU / w_MR | 1/3, 1/3, 1/6, 1/6 | (cung) | (cung) | (cung) |
| lambda_KD | 0 | 0 | 0 | 0 |
| lambda_CE | 0.25 | 0.25 | 0.25 | **0.25** |
| lambda_IHL | 5.0 | 5.0 | 5.0 | **5.0** |
| loku_subtract_scale | 1.5 | 1.5 | 1.5 | **1.0** |
| loku_image_subtract_scale | 1.0 | 1.0 | 1.0 | **1.0** |
| lora_image_last_k_blocks | 2 | 3 | 3 | **3** |
| lora_image_include_fc1 | 0 | 1 | 0 | **0** |


In [ ]:
# Cell 1: setup
import os, subprocess
WORK='/kaggle/working'; REPO=f'{WORK}/Forget-MI-LoKU'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/nhnhu146/Forget-MI-LoKU.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
os.chdir(REPO)
for f in ['training/forgetmi_p3_cand.py','training/eval_multimodal.py','training/adv_common.py']:
    assert os.path.exists(f), f'Thieu {f} -> git push code moi truoc'
subprocess.run(['pip','install','-q','pydicom','scikit-image','scikit-learn','pyyaml','wandb','seaborn==0.13.2'],check=True)
subprocess.run(['pip','install','-q','transformers==4.38.0','peft==0.10.0','accelerate==0.27.0'],check=True)
import torch; assert torch.cuda.is_available(),'Bat GPU'
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GPU   :',torch.cuda.get_device_name(0))

# Tu kiem phan thuan-mang cua eval_multimodal (30 giay, truoc khi ton 2,7h GPU)
print('\n--- tu kiem eval_multimodal ---')
subprocess.run(['python','tools/test_eval_multimodal.py'],check=True)


In [ ]:
# Cell 2: CHON VARIANT + path
import glob, os

VARIANT = 'f6'      # f5 | f6   <- DOI DUNG DONG NAY
SEED    = 42
EPOCHS  = 30

# Nen chung cua ca hai: P3-NoKD-More + luoi tang luc quen.
BASE_OVR = {'lambda_ihl':5.0, 'lambda_ce':0.25,
            'loku_image_subtract_scale':1.0,
            'lora_image_last_k_blocks':3,
            'lora_image_include_fc1':0}
# Khac nhau DUNG mot khoa: he so tru FILA o nhanh van ban (W* = W - gamma*B*A*).
VARIANTS = {'f5': {'loku_subtract_scale':1.5},   # tru qua da 50%, da chay
            'f6': {'loku_subtract_scale':1.0}}   # gia tri chuan LoKU
assert VARIANT in VARIANTS, f'VARIANT phai thuoc {sorted(VARIANTS)}'
OVR_METHOD = {**BASE_OVR, **VARIANTS[VARIANT]}

RID = f'p3_{VARIANT}_s{SEED}'

def fd(*slugs):
    for s in slugs:
        if os.path.isdir(f'/kaggle/input/{s}'): return f'/kaggle/input/{s}'
        h=glob.glob(f'/kaggle/input/datasets/*/{s}')
        if h: return sorted(h)[0]
    return None
def bins(root): return sorted(glob.glob(os.path.join(root,'**','pytorch_model.bin'),recursive=True),key=len)

CONFIG='config_advanced_kaggle.yaml'
DATA=fd('forget-mi-data'); MOD=fd('forget-mi-models-full','forget-mi-models')
assert DATA and MOD,'Add Input: forget-mi-data + forget-mi-models-full'
BASE=os.path.dirname([b for b in bins(MOD) if 'training_original_model' in b][0])
gh=[b for b in bins(MOD) if 'model_retrained_3per' in b]
assert gh,'Khong thay model_retrained_3per'
GOLD=os.path.dirname(gh[0])
TEXT=os.path.join(DATA,'data','metadata'); IMG=os.path.join(DATA,'data','img_data')
SPLIT='./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv'
FORGET='./data_splits/forget_set_3per.csv'
for n,p in {'BASE':BASE,'GOLD':GOLD,'TEXT':TEXT,'IMG':IMG,'SPLIT':SPLIT,'FORGET':FORGET}.items():
    assert p and os.path.exists(p),f'Missing {n}: {p}'

OUT     = f'/kaggle/working/kltn_p3_{VARIANT}_s{SEED}'
OD      = f'{OUT}/{RID}'
CKPT    = f'{OD}/checkpoints/latest.pt'        # E30, adv_common.save_ckpt ghi moi epoch
R_TRAIN = f'/kaggle/working/results_p3_{VARIANT}.csv'
R_MM    = f'/kaggle/working/results_multimodal_{VARIANT}.csv'
HIST    = f'/kaggle/working/perepoch_{RID}.csv'

COMMON={'forget_set_path':FORGET,'base_model_path':BASE,'bert_pretrained_dir':BASE,
        'retrained_model_path':GOLD,'text_data_dir':TEXT,'img_data_dir':IMG,
        'data_split_path':SPLIT,'use_noise':1}
MLP_TXT='attention.output.dense|intermediate.dense|output.dense'
MORE={'lora_extra_target_modules':MLP_TXT}     # P3-NoKD-More, khoa tu MIMIC 3%

# Khoa PHAI khai lai luc dung lai W* tu checkpoint chi-chua-LoRA: hai he so tru FILA khong
# suy duoc tu ten khoa (khac lora_extra / lora_image_last_k_blocks) ma lai quyet dinh
# W* = W - gamma*B*A* -> sai gamma la moi so sai AM THAM, khong assert nao bat.
REBUILD={k:OVR_METHOD[k] for k in ['loku_subtract_scale','loku_image_subtract_scale',
                                   'lora_image_last_k_blocks','lora_image_include_fc1']}

print('VARIANT  :',VARIANT)
print('run id   :',RID)
print('ckpt se o:',CKPT)
print('\n--- overrides cua phuong phap ---')
for k,v in OVR_METHOD.items(): print(f'   {k:32} = {v}')
print('\nSo tham so KHONG phu thuoc he so tru -> phai ra DUNG 1,488,896 (nhu f5).')
print('Lech la cau hinh chua vao dung, tat ngay dung de chay het 2,7h.')


In [ ]:
# Cell 3: HUAN LUYEN (30 epoch, ~2,7h)
import os, subprocess, time
env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled',
     'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}

ovr=dict(COMMON); ovr.update(MORE)
ovr.update({'id':RID,'output_dir':OD,'unlearn_epochs':EPOCHS,'results_csv_path':R_TRAIN,
            'ce_selector':1,'s4_delta':0.15,'history_csv_path':HIST})
ovr.update(OVR_METHOD)
cmd=['python','training/forgetmi_p3_cand.py','--config',CONFIG,'--seed',str(SEED),
     '--scheme','uni_nokd','--ablate','none','--fresh','--override',
     ','.join(f'{k}={v}' for k,v in ovr.items())]

print('='*72+f'\nTRAIN {RID}\n'+'='*72)
t0=time.time()
try:
    subprocess.run(cmd,env=env,check=True)
    print(f'OK train  {(time.time()-t0)/3600:.2f}h')
except subprocess.CalledProcessError as e:
    print('FAIL train rc=',e.returncode)
print('checkpoint ton tai:',os.path.exists(CKPT))


In [ ]:
# Cell 4: EVAL DA NHANH tren checkpoint vua tao (img / txt / fuse)
# Chay FP16 truoc (cung do chinh xac voi og/re/f4 da do). Neu nhanh van ban tran FP16 va
# cho NaN (f3/f5 da dinh) thi TU DONG chay lai FP32, ghi them nhan <VARIANT>_fp32.
# Voi f6 (gamma=1.0) ky vong FP16 chay thang, khong can fallback.
import os, subprocess, time
import pandas as pd

def run_eval(label, extra):
    ovr=dict(COMMON); ovr.update(MORE); ovr.update(REBUILD)
    ovr['output_dir']=f'{OUT}/_mm'; ovr.update(extra)
    cmd=['python','training/eval_multimodal.py','--config',CONFIG,'--seed',str(SEED),
         '--label',label,'--model_type','p3_lora','--model_path',CKPT,
         '--out_csv',R_MM,'--override',','.join(f'{k}={v}' for k,v in ovr.items())]
    print('='*72+f'\nEVAL {label}\n'+'='*72)
    t0=time.time()
    try:
        subprocess.run(cmd,env=env,check=True)
        print(f'OK {label}  {(time.time()-t0)/60:.1f} phut'); return True
    except subprocess.CalledProcessError as e:
        print(f'FAIL {label} rc={e.returncode}'); return False

if not os.path.exists(CKPT):
    print('Khong co checkpoint -> bo qua eval. Xem lai Cell 3.')
else:
    run_eval(VARIANT, {})
    need_fp32=False
    if os.path.exists(R_MM):
        d=pd.read_csv(R_MM)
        d=d[d['label']==VARIANT]
        need_fp32=bool(d[d['view'].isin(['txt','fuse'])]['Df_AUC'].isna().any())
    if need_fp32:
        print('\ntxt/fuse ra NaN o FP16 -> chay lai FP32...')
        run_eval(f'{VARIANT}_fp32', {'eval_autocast':0})
    else:
        print('\nFP16 du dung, khong can chay lai FP32.')
        print('(Voi f6 day la ket qua MONG DOI: bo tru qua da thi het tran so.)')


In [ ]:
# Cell 5: BANG KET QUA + TU KIEM
import os
import pandas as pd
pd.set_option('display.width',250)

print(f'===== HUAN LUYEN {VARIANT} (E30 = checkpoint_kind "last") =====')
train_row=None
if os.path.exists(R_TRAIN):
    dt=pd.read_csv(R_TRAIN)
    cols=[c for c in ['id','checkpoint_kind','selected_epoch','Forget_AUC','Forget_Macro_F1',
                      'Test_AUC','Test_Macro_F1','MIA','MIA_paper','forget_ce','test_ce',
                      '1_minus_Sim','trainable_params','trainable_ratio','core_seconds']
          if c in dt.columns]
    print(dt[cols].to_string(index=False))
    last=dt[dt['checkpoint_kind']=='last']
    if len(last): train_row=last.iloc[-1]
else:
    print('chua co',R_TRAIN)

print('\n===== EVAL DA NHANH =====')
if os.path.exists(R_MM):
    dm=pd.read_csv(R_MM)
    c=['label','view','Df_AUC','Df_F1','Dt_AUC','Dt_F1','MIA','MIA_paper',
       'member_ce','nonmember_ce','forget_ce']
    print(dm[[x for x in c if x in dm.columns]].to_string(index=False))
else:
    print('chua co',R_MM)

print('\n===== TU KIEM =====')
if train_row is not None:
    n=int(train_row['trainable_params'])
    print(f'  tham so {n:,}  ' + ('TRUNG f5 (1,488,896)' if n==1488896
          else '*** LECH f5 -> cau hinh khac, xem lai Cell 2 ***'))
if train_row is not None and os.path.exists(R_MM):
    img=pd.read_csv(R_MM)
    img=img[(img['label']==VARIANT) & (img['view']=='img')]
    if len(img):
        r=img.iloc[-1]
        for name,a,b in [('Df-AUC', r['Df_AUC'], train_row['Forget_AUC']),
                         ('Dt-AUC', r['Dt_AUC'], train_row['Test_AUC']),
                         ('MIA',    r['MIA'],    train_row['MIA']),
                         ('forget-CE', r['forget_ce'], train_row['forget_ce'])]:
            ok = abs(float(a)-float(b)) < 0.005
            print(f'  {name:10} eval {float(a):7.4f}  vs  train {float(b):7.4f}   '
                  + ('TRUNG' if ok else '*** LECH -> dung tin so txt/fuse ***'))

print('\n===== SO SANH (nhanh anh, E30) =====')
print('  mo hinh   Df-AUC   Dt-AUC     MIA  forget-CE  test-CE      tham so')
print('  theta_og   0.731    0.695   0.657      1.966    2.108  113,238,164')
print('  gold       0.498    0.615   0.423      4.736    3.046  113,238,164')
print('  Forget-MI  0.623    0.648   0.627      3.578    2.906  113,238,164')
print('  F3         0.590    0.692   0.418      3.477    2.256    1,451,008')
print('  F4         0.528    0.667   0.368      4.506    2.369    1,495,072')
print('  F5         0.522    0.670   0.279      4.498    2.360    1,488,896')
print('  --> F6 ~ F5 : chot F6 (cau hinh sach, gamma=1.0 chuan LoKU).')
print('  --> F6 kem  : giu F5, gamma=1.5 la phat hien that, phai giai thich.')

print(f'\nTAI VE: results_p3_{VARIANT}.csv - results_multimodal_{VARIANT}.csv - perepoch_{RID}.csv')
print(f'        + kltn_p3_{VARIANT}_s{SEED}/**/checkpoints/latest.pt')
